# PlantDoc Scratch Models: Final Holdout

This notebook evaluates the five frozen random-initialization, three-seed scratch comparisons exactly once. It starts in preflight mode and does not run final-holdout inference until explicitly unlocked.

In [1]:
# Use preflight first. Sealed evaluation is a one-time final comparison.
RUN_MODE = 'sealed_evaluation'  # preflight | sealed_evaluation | report_existing
ACKNOWLEDGEMENT = 'RUN_PLANTDOC_SCRATCH_FINAL_HOLDOUT_ONCE'
SEALED_ACKNOWLEDGEMENT = 'RUN_PLANTDOC_SCRATCH_FINAL_HOLDOUT_ONCE'

DEVICE = 'cpu'
BATCH_SIZE = 64
NUM_WORKERS = 0


In [2]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'code' / 'research_evaluation' / 'sealed_scratch.py').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook inside the Agrivision4 workspace')

REPO_ROOT = find_repo_root(Path.cwd())
CODE_ROOT = REPO_ROOT / "code"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
print({'repo_root': str(REPO_ROOT), 'run_mode': RUN_MODE, 'device': DEVICE})


{'repo_root': '/run/media/sha-weed/adc6c87e-0116-4a25-a6e9-8555193c1e51/Research/Agrivision4', 'run_mode': 'sealed_evaluation', 'device': 'cpu'}


## Execute

In final mode, the evaluator verifies the frozen manifest/checkpoint contract, then writes all results atomically. Progress is printed after each of the 15 checkpoint evaluations.

In [3]:
from research_evaluation.sealed_scratch import build_spec, execute

spec = build_spec(REPO_ROOT, 'plantdoc')
result = execute(
    spec,
    run_mode=RUN_MODE,
    acknowledgement=ACKNOWLEDGEMENT,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print({'status': result['status'], 'sealed_inference': result.get('sealed_inference_executed', False)})


/home/sha-weed/.local/share/mamba/envs/.venv_ensemble/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{"event": "checkpoint_evaluated", "index": 1, "total": 15, "model_key": "wavestarnet_big_transfer_v2", "seed": 42, "accuracy": 0.4744186046511628, "macro_f1": 0.4361611349028749}
{"event": "checkpoint_evaluated", "index": 2, "total": 15, "model_key": "wavestarnet_big_transfer_v2", "seed": 2024, "accuracy": 0.4790697674418605, "macro_f1": 0.43233267466178293}
{"event": "checkpoint_evaluated", "index": 3, "total": 15, "model_key": "wavestarnet_big_transfer_v2", "seed": 3407, "accuracy": 0.4906976744186046, "macro_f1": 0.44135827244496517}
{"event": "checkpoint_evaluated", "index": 4, "total": 15, "model_key": "efficientnet_b0", "seed": 42, "accuracy": 0.39069767441860465, "macro_f1": 0.34361194773804477}
{"event": "checkpoint_evaluated", "index": 5, "total": 15, "model_key": "efficientnet_b0", "seed": 2024, "accuracy": 0.37906976744186044, "macro_f1": 0.3200174876118109}
{"event": "checkpoint_evaluated", "index": 6, "total": 15, "model_key": "efficientnet_b0", "seed": 3407, "accuracy": 0

## Principal Results

The tables below remain embedded after you save the executed notebook.

In [4]:
import pandas as pd
from IPython.display import Markdown, display

if RUN_MODE == 'preflight':
    protocol = result['protocol']
    display(Markdown('### Preflight passed: no sealed inference executed'))
    display(pd.DataFrame([{
        'final_holdout_samples': protocol['test_samples'],
        'classes': protocol['class_count'],
        'minimum_class_support': protocol['minimum_class_support'],
        'checkpoint_count': result['checkpoint_count'],
        'strict_loads': len(result['strict_load_audits']),
    }]))
    display(pd.DataFrame(result['checkpoint_table'])[['display_name', 'model_key', 'seed', 'parameters', 'checkpoint_sha256']])
else:
    display(Markdown('### Final aggregate ranking: mean, SD, and 95% t CI'))
    display(result['aggregate'])
    display(Markdown('### Per-seed final-holdout metrics'))
    display(result['seed_metrics'])
    if 'artifact_inventory' in result:
        display(Markdown(f"### Output inventory: {len(result['artifact_inventory'])} files"))
        display(pd.DataFrame(result['artifact_inventory']))
    display(Markdown(f"**Saved to:** `{result['output_directory']}`"))


### Final aggregate ranking: mean, SD, and 95% t CI

,rank_by_macro_f1,model_key,display_name,parameters,seeds,seed_count,accuracy_mean,accuracy_std,accuracy_ci95_low,accuracy_ci95_high,...,nll_ci95_low,nll_ci95_high,brier_mean,brier_std,brier_ci95_low,brier_ci95_high,ece_15bin_mean,ece_15bin_std,ece_15bin_ci95_low,ece_15bin_ci95_high
0,1,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,2960615,"42,2024,3407",3,0.481395,0.008385,0.460566,0.502225,...,1.702545,1.863743,0.654468,0.005465,0.640891,0.668044,0.088744,0.006285,0.073131,0.104357
1,2,mobilevit_s,MobileViT-S,4954939,"42,2024,3407",3,0.477519,0.024428,0.416837,0.538201,...,1.843406,2.054846,0.693917,0.018273,0.648525,0.739309,0.152779,0.016236,0.112446,0.193111
2,3,mobilenetv3_large_100,MobileNetV3-Large,4236619,"42,2024,3407",3,0.406977,0.022185,0.351867,0.462086,...,2.389214,2.623388,0.794553,0.021270,0.741717,0.847390,0.179036,0.004690,0.167386,0.190686
3,4,efficientnet_b3,EfficientNet-B3,10737731,"42,2024,3407",3,0.404651,0.017558,0.361035,0.448267,...,2.250780,2.699368,0.803935,0.006398,0.788041,0.819830,0.228081,0.011142,0.200402,0.255760
4,5,efficientnet_b0,EfficientNet-B0,4042135,"42,2024,3407",3,0.398450,0.024205,0.338320,0.458579,...,2.476167,2.703451,0.809943,0.014722,0.773372,0.846513,0.215870,0.018996,0.168682,0.263058


### Per-seed final-holdout metrics

,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,weighted_precision,weighted_recall,nll,brier,ece_15bin,model_key,display_name,seed,parameters,runtime_seconds,checkpoint_sha256
0,0.474419,0.451997,0.436161,0.458921,0.441756,0.451997,0.460474,0.474419,1.814766,0.657160,0.093716,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,42,2960615,24.995800,f2109ac1994687d5c29473f8fd9afeb60908776e3491f2...
1,0.479070,0.449665,0.432333,0.458157,0.439983,0.449665,0.458729,0.479070,1.784732,0.648178,0.090837,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,2024,2960615,25.610164,2281787da67db4a320bb48c25f34a8e171e1b807bbd72c...
2,0.490698,0.447131,0.441358,0.477149,0.451099,0.447131,0.480811,0.490698,1.749933,0.658064,0.081680,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,3407,2960615,24.681514,326fb59033400a2afcddcdd681e47103192d05bb38e87b...
3,0.390698,0.363338,0.343612,0.370847,0.346095,0.363338,0.369676,0.390698,2.641912,0.812475,0.211836,efficientnet_b0,EfficientNet-B0,42,4042135,18.045941,b3226958c783ed91c738d4d86a5fec5ac54deebda263eb...
4,0.379070,0.323835,0.320017,0.358581,0.337531,0.323835,0.360380,0.379070,2.556224,0.823234,0.236558,efficientnet_b0,EfficientNet-B0,2024,4042135,18.987612,364c3c2ce7a7909754b1afae8b63f91804ff77b796116c...
5,0.425581,0.377556,0.365636,0.406721,0.378557,0.377556,0.409076,0.425581,2.571291,0.794120,0.199215,efficientnet_b0,EfficientNet-B0,3407,4042135,19.291444,26755f20657b1e7fde8299c5b624b4dde06387a2049e1d...
6,0.406977,0.364644,0.365805,0.393569,0.394759,0.364644,0.400727,0.406977,2.403455,0.797089,0.215389,efficientnet_b3,EfficientNet-B3,42,10737731,32.823145,7833fea2455be0fee3fe7a0790dd9f035f29cbfbb110ab...
7,0.420930,0.379098,0.378049,0.413376,0.391832,0.379098,0.419920,0.420930,2.576499,0.804952,0.232602,efficientnet_b3,EfficientNet-B3,2024,10737731,31.515011,4f305256e1426f79afb99ce8d282e4d80e2d75081b6809...
8,0.386047,0.329190,0.318750,0.361391,0.324653,0.329190,0.355463,0.386047,2.445267,0.809764,0.236252,efficientnet_b3,EfficientNet-B3,3407,10737731,31.590024,319caa74d9340eed70cddac7ec04608bd817005e076a62...
9,0.432558,0.395500,0.380349,0.411432,0.395769,0.395500,0.421809,0.432558,2.451876,0.771732,0.174530,mobilenetv3_large_100,MobileNetV3-Large,42,4236619,14.539530,f5eaeffe1d37edc35c24caf7e617f09e720972ac8eb9f9...


**Saved to:** `/run/media/sha-weed/adc6c87e-0116-4a25-a6e9-8555193c1e51/Research/Agrivision4/plantdoc/results/sealed_test/plantdoc_scratch_final_holdout_v1`

If final evaluation completes but the notebook is not saved, set `RUN_MODE = 'report_existing'` and rerun all cells. That mode checks every published artifact hash and reloads the tables without reading images or running inference.